# Agentic retrieval demo

Calls the local `/retrieve` endpoint (see `docker/README.md`) and passes the
GigaSearch pipeline config explicitly via `search_params.configuration`.

The config is loaded from `configs/search_config.v.0.2.4.json` — the same file
referenced by `search.gigasearch.config_path` in `configs/ift_inference.yaml`.

Prereq: agent service running, e.g.

```bash
AGENT_CONFIG=configs/ift_inference.yaml PYTHONPATH=. \
  .venv/bin/python -m uvicorn agent.service:app --host 127.0.0.1 --port 8090
```

In [1]:
import json
from pathlib import Path

import requests
import yaml

ROOT = Path("..").resolve()
AGENT_CONFIG_PATH = ROOT / "configs/ift_inference.yaml"
SEARCH_CONFIG_PATH = ROOT / "configs/search_config.v.0.2.4.json"
AGENT_URL = "http://127.0.0.1:8090"

In [16]:
with AGENT_CONFIG_PATH.open(encoding="utf-8") as f:
    agent_cfg = yaml.safe_load(f)

with SEARCH_CONFIG_PATH.open(encoding="utf-8") as f:
    gigasearch_configuration = json.load(f)

index_id = (
    gigasearch_configuration.get("agent_configuration", {})
    .get("data_sources_faq", [{}])[0]
    .get("index_id")
)
embedder = (
    gigasearch_configuration.get("agent_configuration", {})
    .get("retriever_faq", {})
    .get("embedder", {})
    .get("model_name")
)

print(f"search config: {SEARCH_CONFIG_PATH.name}")
print(f"index_id:      {index_id}")
print(f"embedder:      {embedder}")
print(f"agent default: {agent_cfg['search']['gigasearch'].get('config_path')}")

search config: search_config.v.0.2.4.json
index_id:      1a0cbc8d-5df4-4bb2-abda-b4bdddfd708d_search_usooper_3
embedder:      gigachat-3b
agent default: configs/search_config.v.0.2.4.json


### `search_params`

`configuration` is peeled off by `GigaSearchClient.local_search` and sent as
the top-level GigaSearch `configuration` field (same as `scripts/DEMO.ipynb`).
Any other keys (`filters`, …) are merged into the inner chat `request`.

**Why you might still see 3 hits when `size` is 8**

Two separate limits apply:

| Knob | Where | Effect |
|------|-------|--------|
| `retriever_faq.size` | `search_config.*.json` → GigaSearch `configuration` | How many passages GigaSearch returns |
| `top_k_default` | `configs/ift_inference.yaml` → harness | Max hits shown to the model per search call |

`num_results` in the trajectory = `min(harness_top_k, gigasearch_raw_hits)`.

If `retriever_size=3` appears in the trajectory after you edited the json,
the running agent is still using the old config — either restart the service
or pass `search_params.configuration` (this notebook does the latter).

After restarting the agent, each search `result_summary` includes
`search_config`, `search_config_source`, `harness_top_k`, `gigasearch_raw_hits`.

In [17]:
search_params = {
    "configuration": gigasearch_configuration,
    # Optional per-request filters (see scripts/DEMO.ipynb):
    # "filters": {
    #     "operator": "and",
    #     "elements": [
    #         {"field": "breadcrumbs", "value": "*капремонт*", "operator": "wildcard"}
    #     ],
    # },
}

question = "Когда исправили ошибку с отсутствием документов по спецсчету капремонта?"
prompt_version = agent_cfg.get("agent", {}).get("prompt_version", "v2_search_only")

payload = {
    "messages": [{"role": "user", "content": question}],
    "search_params": search_params,
    "prompt_version": prompt_version,
    "include_trajectory": True,
}

payload

{'messages': [{'role': 'user',
   'content': 'Когда исправили ошибку с отсутствием документов по спецсчету капремонта?'}],
 'search_params': {'configuration': {'cache': {'ignore_cache': True},
   'agent_type': 'universal',
   'agent_configuration': {'index_type': 'opensearch',
    'data_sources_faq': [{'index_id': '1a0cbc8d-5df4-4bb2-abda-b4bdddfd708d_search_usooper_3'}],
    'query_rewriter': {'enabled': False},
    'retriever_faq': {'global_size': 16,
     'size': 8,
     'bm25_weight': 0.25,
     'embedder': {'model_name': 'gigachat-3b'},
     'search_query_enricher': {'enabled': False,
      'rag_fusion': {'enabled': False,
       'number_of_generated_questions': 3,
       'model_name': 'GigaChat-2-Max'}}},
    'reranker_faq': {'enabled': False},
    'context_expansion': {'enabled': False,
     'after_passages': 2,
     'max_tokens': 8000},
    'qa': {'enabled': False,
     'qa_scenario': 'stuff',
     'model_endpoint': 'default',
     'model_name': 'GigaChat-2-Max',
     'context_

In [21]:
response = requests.post(
    f"{AGENT_URL}/retrieve",
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
response.raise_for_status()
result = response.json()
result

{'ranked_doc_ids': ['aea8b6f4-76ea-487f-8b3a-5cce1743a5b1'],
 'ranked_passages': [{'doc_id': 'aea8b6f4-76ea-487f-8b3a-5cce1743a5b1',
   'title': 'Общее пространство ЦКР (УСО, ТП, УУП)/Дефекты/Закрытые дефекты/СББОЛ/Отсутствуют ранее вложенные документы по спец. счёту капитального ремонта (DCBHCK6-12464) - ЗАКРЫТ',
   'text': '| Номер дефекта | DCBHCK6-12464 |\n| --- | --- |\n| Статус дефекта | Закрыт |\n| Описание | Отсутствуют ранее вложенные документы по спец. счёту капитального ремонта. У клиента до мая 2023 документы отсутствуют, с мая по текущий период документы отображаются. |\n| Обходное решение | Отсутствует |\n| Дата регистрации дефекта | 07.08.2023 |\n| Срок исправления / Номер версии (патча) | 09.08.2023 |\n| Ответ | {{UL\\_MARKER}} Ошибка действительно есть, приношу извинения … (рекомендации из описания ошибки); {{UL\\_MARKER}} Сейчас прорабатываются возможные варианты решения вопроса ; {{UL\\_MARKER}} Банк работает над исправлением. Приношу извинения за эту ситуацию. Сроки

In [10]:
def show_search_validation(result: dict) -> None:
    traj = result.get("trajectory") or {}
    for tc in traj.get("tool_calls") or []:
        if tc.get("tool") != "search":
            continue
        rs = tc.get("result_summary") or {}
        cfg = rs.get("search_config") or {}
        print(
            f"turn {tc.get('turn')}: "
            f"config_source={rs.get('search_config_source')} "
            f"retriever_size={cfg.get('retriever_size')} "
            f"global_size={cfg.get('retriever_global_size')} | "
            f"harness_top_k={rs.get('harness_top_k')} "
            f"gigasearch_raw_hits={rs.get('gigasearch_raw_hits')} "
            f"shown_to_model={rs.get('num_results')}"
        )
        if cfg.get("retriever_size") != rs.get("num_results"):
            print(
                "  note: num_results is min(harness_top_k, gigasearch_raw_hits); "
                "retriever_size is what GigaSearch was asked to return"
            )


print("ranked_doc_ids:", result.get("ranked_doc_ids"))
print("stopped_reason:", result.get("stopped_reason"))
print("num_tool_calls:", result.get("num_tool_calls"))
print()
show_search_validation(result)

for i, p in enumerate(result.get("ranked_passages") or []):
    print(f"\n--- rank {i} ---")
    print("doc_id:", p.get("doc_id"))
    print("title:", (p.get("title") or "")[:120])
    print("text:", (p.get("text") or "")[:300], "...")

traj = result.get("trajectory") or {}
search_latencies = [
    tc["latency_ms"]
    for tc in (traj.get("tool_calls") or [])
    if tc.get("tool") == "search" and tc.get("latency_ms") is not None
]
if search_latencies:
    print(f"\nsearch latency ms: {search_latencies} (total {sum(search_latencies)})")

ranked_doc_ids: ['aea8b6f4-76ea-487f-8b3a-5cce1743a5b1', 'fb6c7b7e-745e-4057-a731-7bb669486217', 'f223fc96-625f-4fa2-bb21-689aa689e7cb']
stopped_reason: answer
num_tool_calls: 3

--- rank 0 ---
doc_id: aea8b6f4-76ea-487f-8b3a-5cce1743a5b1
title: Общее пространство ЦКР (УСО, ТП, УУП)/Дефекты/Закрытые дефекты/СББОЛ/Отсутствуют ранее вложенные документы по спец. счёт
text: | Номер дефекта | DCBHCK6-12464 |
| --- | --- |
| Статус дефекта | Закрыт |
| Описание | Отсутствуют ранее вложенные документы по спец. счёту капитального ремонта. У клиента до мая 2023 документы отсутствуют, с мая по текущий период документы отображаются. |
| Обходное решение | Отсутствует |
| Дата ...

--- rank 1 ---
doc_id: fb6c7b7e-745e-4057-a731-7bb669486217
title: Общее пространство ЦКР (УСО, ТП, УУП)/Банковские счета (УСО, ТП, УУП)/Специальные банковские счета (УСО, ТП, УУП)/Счета 
text: Обосновывающие документы предоставляются в банк владельцем спец.счёта вместе с платежом в ВСП или прикладываются к платежу в Сб

### CKR eval

The eval runner loads `configs/search_config.v.0.2.4.json` by default — no need to
pass `--search-params` manually:

```bash
# from repo root
PYTHONPATH=. bash scripts/run_ckr_eval.sh

# or directly
PYTHONPATH=. python3 -m eval_.run_ckr_eval --all
```

Use a different search config:

```bash
PYTHONPATH=. python3 -m eval_.run_ckr_eval --all \
  --search-config configs/search_config.v.0.2.4.json
```

Smoke test on 5 questions:

```bash
SUBSET_SIZE=5 PYTHONPATH=. bash scripts/run_ckr_eval.sh
```